In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "backend" / "data"
INPUT_PATH = DATA_DIR / "WELFake.csv"
OUTPUT_PATH = DATA_DIR / "WELFake_cleaned.csv"

df = pd.read_csv(INPUT_PATH)

print("Original shape:", df.shape)
display(df.head())

Original shape: (72134, 3)


,title,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,NaN,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [3]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (
        df.isnull().mean() * 100
    ).round(2)
})

display(missing_summary)

,Missing_Count,Missing_Percentage
title,558,0.77
text,39,0.05
label,0,0.00


In [4]:
before = len(df)

df = df.drop_duplicates().copy()
df = df.reset_index(drop=True)

print("Rows before:", before)
print("Rows after:", len(df))
print("Exact duplicates removed:", before - len(df))

Rows before: 72134
Rows after: 63678
Exact duplicates removed: 8456


In [5]:
required_columns = {"title", "text", "label"}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# Keep only rows with valid binary labels.
before = len(df)

print("Rows removed due to invalid labels:", before - len(df))

# Normalize missing title and text fields.
df["title"] = df["title"].fillna("").astype(str).str.strip()
df["text"] = df["text"].fillna("").astype(str).str.strip()

# Remove records with neither a title nor article body.
before = len(df)

df = df[
    df["title"].ne("") | df["text"].ne("")
].copy()

print("Rows with no usable title or text removed:", before - len(df))

Rows removed due to invalid labels: 0
Rows with no usable title or text removed: 0


In [6]:
print("Cleaned shape:", df.shape)

print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing_Count"))

print("\nLabel distribution:")
display(df["label"].value_counts().sort_index().to_frame("Count"))

print("\nExact duplicate rows remaining:", df.duplicated().sum())

print("\nEmpty titles:", df["title"].eq("").sum())
print("Empty article bodies:", df["text"].eq("").sum())

display(df.head())

Cleaned shape: (63678, 3)

Missing values:


,Missing_Count
title,0
text,0
label,0



Label distribution:


,Count
label,
0,34791
1,28887



Exact duplicate rows remaining: 2

Empty titles: 518
Empty article bodies: 568


,title,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last n...",1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [7]:
# Remove any remaining exact duplicates
before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

print("Additional duplicates removed:", before - len(df))
print("Exact duplicates remaining:", df.duplicated().sum())

Additional duplicates removed: 2
Exact duplicates remaining: 0


In [8]:
# Find records where both title and text are empty
empty_records = df[
    df["title"].eq("") & df["text"].eq("")
]

print("Records with both title and text empty:", len(empty_records))

Records with both title and text empty: 0


In [9]:
df = df[
    df["title"].ne("") | df["text"].ne("")
].copy()

df = df.reset_index(drop=True)

print("Final cleaned shape:", df.shape)
print("Missing values:\n", df.isnull().sum())
print("Exact duplicates:", df.duplicated().sum())
print("\nLabel distribution:\n", df["label"].value_counts().sort_index())

Final cleaned shape: (63676, 3)
Missing values:
 title    0
text     0
label    0
dtype: int64
Exact duplicates: 0

Label distribution:
 label
0    34790
1    28886
Name: count, dtype: int64


In [10]:
original_df = pd.read_csv(INPUT_PATH)

print("Original label counts:")
print(original_df["label"].value_counts().sort_index())

print("\nLabel counts after exact deduplication:")
deduplicated_df = original_df.drop_duplicates()

print(deduplicated_df["label"].value_counts().sort_index())

print("\nRows removed by exact deduplication:")
print(len(original_df) - len(deduplicated_df))

Original label counts:
label
0    35028
1    37106
Name: count, dtype: int64

Label counts after exact deduplication:
label
0    34791
1    28887
Name: count, dtype: int64

Rows removed by exact deduplication:
8456


#### Find repeated articles with conflicting labels

In [11]:
# Load the original dataset for investigation
original_df = pd.read_csv(INPUT_PATH)

# Find identical title/text pairs
article_groups = (
    original_df
    .groupby(["title", "text"], dropna=False)
    .agg(
        occurrences=("label", "size"),
        unique_labels=("label", "nunique")
    )
    .reset_index()
)

# Article content appearing with multiple labels
conflicting_articles = article_groups[
    article_groups["unique_labels"] > 1
]

print("Unique title/text combinations:", len(article_groups))

print(
    "Repeated title/text combinations:",
    (article_groups["occurrences"] > 1).sum()
)

print(
    "Articles with conflicting labels:",
    len(conflicting_articles)
)

display(conflicting_articles.head(10))

Unique title/text combinations: 63678
Repeated title/text combinations: 7903
Articles with conflicting labels: 0


,title,text,occurrences,unique_labels


In [12]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print("Cleaned dataset saved!")
print("Location:", OUTPUT_PATH.resolve())
print("Shape:", df.shape)

Cleaned dataset saved!
Location: C:\Projects\fake_news_detection_extension\backend\data\WELFake_cleaned.csv
Shape: (63676, 3)
